In [ ]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F

CATALOGO = "desafio_vendas"
CAMINHO_ORIGEM = "/Volumes/desafio_vendas/bronze/origem"
NOME_ARQUIVO = "Dataset_vendas_1.csv"
TABELA_BRONZE = f"{CATALOGO}.bronze.vendas_raw"
TABELA_LOG = f"{CATALOGO}.ops.log_pipeline"

ID_LOTE = str(uuid.uuid4())
INICIO = datetime.now()

# Remove a máscara de coluna (se existir) para a Bronze poder ser regravada
for c in ["cliente_cpf", "cliente_telefone", "cliente_email"]:
    try:
        spark.sql(f"ALTER TABLE {TABELA_BRONZE} ALTER COLUMN {c} DROP MASK")
    except Exception:
        pass

In [ ]:
MAPA_COLUNAS = {
    "Produto": "produto",
    "Data da venda": "data_venda",
    "Data da retirada": "data_retirada",
    "Valor do frete": "valor_frete",
    "Status de envio": "status_envio",
    "Cidade": "cidade",
    "Estado": "estado",
    "Cliente": "cliente",
    "CPF do cliente": "cliente_cpf",
    "Sexo do cliente": "cliente_sexo",
    "Tipo de produto": "tipo_produto",
    "Quantidade vendida": "qtd_vendida",
    "Preço unitário": "valor_unitario",
    "Status da venda": "status_venda",
    "Tipo de pagamento": "tipo_pagamento",
    "Avaliação do produto": "avaliacao_produto",
    "Status de desconto": "status_desconto",
    "Porcentagem de desconto": "desconto_pct",
    "Item promocional": "item_promocional",
    "Telefone de contato": "cliente_telefone",
    "E-mail de contato": "cliente_email",
}

In [ ]:
caminho = f"{CAMINHO_ORIGEM}/{NOME_ARQUIVO}"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("encoding", "UTF-8")
    .option("sep", ",")
    .csv(caminho)
)

colunas_esperadas = set(MAPA_COLUNAS.keys())
colunas_recebidas = set(df.columns)
assert colunas_esperadas == colunas_recebidas, (
    f"Schema diferente do esperado. Faltando: {colunas_esperadas - colunas_recebidas} | "
    f"Extras: {colunas_recebidas - colunas_esperadas}"
)

for antigo, novo in MAPA_COLUNAS.items():
    df = df.withColumnRenamed(antigo, novo)

df = (
    df
    .withColumn("_arquivo_origem", F.lit(NOME_ARQUIVO))
    .withColumn("_data_ingestao", F.current_timestamp())
    .withColumn("_id_lote", F.lit(ID_LOTE))
    .withColumn("_numero_linha_origem", F.monotonically_increasing_id())
)

In [ ]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TABELA_BRONZE)
)

qtd_lida = df.count()
qtd_gravada = spark.table(TABELA_BRONZE).count()

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA_LOG} (
  id_lote STRING, etapa STRING, tabela_destino STRING,
  qtd_lida BIGINT, qtd_gravada BIGINT, status STRING,
  inicio TIMESTAMP, fim TIMESTAMP
)
""")

log = spark.createDataFrame(
    [(ID_LOTE, "bronze", TABELA_BRONZE, qtd_lida, qtd_gravada,
      "SUCESSO" if qtd_lida == qtd_gravada else "DIVERGENCIA",
      INICIO, datetime.now())],
    "id_lote string, etapa string, tabela_destino string, qtd_lida long, qtd_gravada long, status string, inicio timestamp, fim timestamp"
)
log.write.mode("append").saveAsTable(TABELA_LOG)

display(spark.table(TABELA_LOG))

In [ ]:
spark.table(TABELA_BRONZE).printSchema()
display(spark.table(TABELA_BRONZE).limit(10))

In [ ]:
import json
dbutils.notebook.exit(json.dumps({
    "id_lote": ID_LOTE, "qtd_lida": qtd_lida, "qtd_gravada": qtd_gravada,
    "qtd_rejeitada": 0, "qtd_duplicada": 0,
    "status": "SUCESSO" if qtd_lida == qtd_gravada else "DIVERGENCIA"}))